# Cal Field Hockey Win Predictor
## Can I predict whether Cal will win based on in-game statistics?
This project uses 11 seasons of Cal field hockey data (2015-2025) 
to build a machine learning model that predicts game outcomes based on 
penalty corners, shots on goal, and goalkeeper saves.

## Data Collection
Using Selenium to scrape game-by-game stats from calbears.com across 11 seasons

In [ ]:
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd

# headless chrome driver
options = webdriver.ChromeOptions()
options.add_argument('--headless')
driver = webdriver.Chrome(options=options)

# url of cal field hockey stats
url = "https://calbears.com/sports/field-hockey/stats"
driver.get(url)
time.sleep(3)

# dismiss the cookie consent popup
try:
    consent_button = WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Accept')]"))
    )
    consent_button.click()
    print("dismissed cookie popup")
    time.sleep(1)
except:
    print("no popup found, continuing")

# click game by game tab
game_by_game_tab = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.XPATH, "//a[@href='#game']"))
)
driver.execute_script("arguments[0].click();", game_by_game_tab)
time.sleep(3)
print("clicked game by game tab")

# click comparison sub tab
comparison_tab = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.XPATH, "//a[@href='#game-comparison']"))
)
driver.execute_script("arguments[0].click();", comparison_tab)
time.sleep(5)
print("clicked comparison tab")

# parse the page
soup = BeautifulSoup(driver.page_source, 'html.parser')

# find all tables on the page and print how many there are
tables = soup.find_all('table')
print(f"found {len(tables)} tables")

table_num = 0
# the comparison table is the one with 'Corners' in the header
for i, table in enumerate(tables):
    headers = [th.text.strip() for th in table.find_all('th')]
    if 'Corners' in headers:
        print(f"comparison table is table #{i}")
        table_num = i
        for row in table.find_all('tr'):
            cols = [ele.text.strip() for ele in row.find_all(['td','th'])]
            if cols:
                print(cols)
        break

## Exploratory Data Analysis
Visualizing the relationship between each stat and game outcomes

In [ ]:
import pandas as pd

df = pd.read_csv('/Users/adya/Desktop/cal_fh_data.csv')
print(df.shape)
print(df.head())


In [ ]:
print(df.isnull().sum())

In [ ]:
print(df.describe()) 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# create a blank canvas with subplots
# 1 row, 3 columns, figure size 12 wide by 5 tall
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sns.boxplot(x="result", y="corners", data=df, ax=axes[0])
axes[0].set_title('corners: wins v losses')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Loss', 'Win'])

#saves to wins
sns.boxplot(x="result", y="saves", data=df, ax=axes[1])
axes[1].set_title('saves: wins v losses')
axes[1].set_xticks([0,1])
axes[1].set_xticklabels(['Loss', 'Win'])

#shots to wins
sns.boxplot(x="result", y="shots_on_goal", data=df, ax=axes[2])
axes[2].set_title('shots: wins v losses')
axes[2].set_xticks([0,1])
axes[2].set_xticklabels(['Loss', 'Win'])

# show the plot
plt.show()

In [ ]:
sns.heatmap(df[['corners', 'shots_on_goal', 'saves', 'result']].corr(), annot=True)

## EDA Findings

The correlation heatmap reveals that shots on goal(sog) is the strongest predictor of winning with a correlation of 0.48, followed by saves (0.31) and corners (0.25). 

Shots on goal and saves have a 0.92 correlation with each other. This makes sense because shots on goal measure Cal's offensive output while saves measure how much defensive pressure Cal is under.
In more evenly matched, high-intensity games, both Cal and the opposing team will tend to be more active, driving both stats up simultaneously.

Corners showed the weakest relationship with winning, suggesting that creating scoring opportunities through shots in regular game play matters more.

## Model Building
Training Logistic Regression and Decision Tree classifiers

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('/Users/adya/Desktop/cal_fh_data.csv')


# define features and target
X = df[['corners', 'shots_on_goal', 'saves']]
y = df['result']

# split 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training games:", len(X_train))
print("Testing games:", len(X_test))

In [ ]:
# training the model
from sklearn.linear_model import LogisticRegression
linear_model =  LogisticRegression()
linear_model.fit(X_train, y_train)

In [ ]:
#testing the model 
from sklearn.metrics import accuracy_score
prediction_y = linear_model.predict(X_test)
print(accuracy_score(prediction_y, y_test))
#accuracy score is the percent of predictions the model got right

In [ ]:
from sklearn.tree import DecisionTreeClassifier
descion_tree = DecisionTreeClassifier(max_depth=3)
descion_tree.fit(X_train, y_train)


In [ ]:
#testing the model 
prediction_y_tree = descion_tree.predict(X_test)
print(accuracy_score(prediction_y_tree, y_test))



In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(20, 10))
plot_tree(descion_tree, feature_names=['corners', 'shots_on_goal', 'saves'], class_names=['Loss', 'Win'], filled=True)
plt.show()

## Evaluation

Rule 1 — Shots on Goal first:

If SOG ≤ 7.5 → likely Loss (90 games, mostly losses)
If SOG > 7.5 → likely Win (58 games, mostly wins)

Rule 2 — If SOG ≤ 7.5:

If SOG ≤ 2.5 → Loss (18 out of 18, perfect!)
If SOG > 2.5 but saves ≤ 1.5 → mostly Loss
If SOG > 2.5 and saves > 1.5 → still mostly Loss

Rule 3 — If SOG > 7.5:

If saves ≤ 5.5 → Win (13 out of 13, perfect!)
If saves > 5.5 and corners ≤ 11.5 → Win (19 out of 19, perfect!)

True Negative — model predicted Loss, it was actually a Loss ✅
True Positive — model predicted Win, it was actually a Win ✅
False Positive — model predicted Win, it was actually a Loss ❌
False Negative — model predicted Loss, it was actually a Win ❌


In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, prediction_y)

In [ ]:
sns.heatmap(data=cm, xticklabels=['Loss', 'Win'], yticklabels=['Loss', 'Win'],  annot=True, fmt='d')

In [ ]:
# bar graph
features = ['corners', 'shots_on_goal', 'saves']
coefficients = linear_model.coef_[0]
plt.bar(features, coefficients)
plt.xlabel('Features')
plt.ylabel('Coefficient Values')
plt.title('How influential each feature is on if cal will win')
plt.show()



This data analysis has shown that the most important factor when understanding when cal will win is SOG( shots on goal), having at least 8 shots on goal is a heavy indicator that Cal will win the game, the next important part is keeping our opponents SOG low, which in turn means keeping our saves down. Corners have little effect. 

## Model Application & Predictions

### Win Probability Predictor
Given in-game stats, what is Cal's probability of winning?

In [ ]:
#take in stats and predict cal's win probability
def cal_win_probability():
    corners = int(input("How many corners did Cal have?"))
    sog = int(input("How many shots on goal did Cal have?"))
    saves = int(input("How many saves did Cal have?"))
    raw_prob = linear_model.predict_proba([[corners, sog, saves]])
    percent_win = raw_prob[0][1]
    return f"Cal is {percent_win:.1%} likely to win, based on the stats you have inputted and their historical data."
cal_win_probability()


In [ ]:
# finding the optimal sog

def unrealistic_best_stats_to_win():
    max_sog = df['shots_on_goal'].max()
    max_saves = df['saves'].max()
    max_corners = df['corners'].max()
    best_prob = 0
    optimal_corners = 0
    optimal_sog = 0
    optimal_saves = 0
    for sog in range(1, max_sog):
        for saves in range(1, max_saves):
            for corners in range(1, max_corners):
                current_prob = linear_model.predict_proba([[corners, sog, saves]])[0][1]
                if current_prob > best_prob:
                    optimal_corners = corners
                    optimal_sog = sog
                    optimal_saves = saves
                    best_prob = current_prob
    return f"The unrealistic but best stats that give Cal the best chance of winning are when Cal has {optimal_corners} corners, {optimal_sog} shots on goal, and {optimal_saves} saves. These stats give them a  {best_prob:.1%} chance of winning."
            


In [ ]:
unrealistic_best_stats_to_win()

In [ ]:
def stats_for_90_percent():
    results = []
    for sog in range(1, 15):
        for saves in range(1, 10):
            for corners in range(1, 12):
                current_prob = linear_model.predict_proba([[corners, sog, saves]])[0][1]
                if current_prob >= 0.9:
                    results.append([corners, sog, saves])
    return results
print(stats_for_90_percent())

In [ ]:
#depending on saves
def stats_for_90_percent_based_on_saves():
    for saves in range(1, 10):
        for sog in range(1, 15):
            current_prob = linear_model.predict_proba([[df['corners'].mean(), sog, saves]])[0][1]
            if current_prob >= 0.90:
                print(f"With {saves} saves, Cal needs {sog} SOG to hit 90% win probability")
                break

stats_for_90_percent_based_on_saves()

## Conclusion


Through the data analysis and models, especially the confusion matrix, I was able to discern that the most influential stat to determine if Cal wins or loses is shots on goal. 


### Model Application

Using the linear regression model I have made, I have built a function to determine whether Cal will win or lose based on stats that the user can input, in order to aid the coaching staff in finding quantitative stats to reach for and rely on. 

It is integral to remember that these stats are just the quantitative side to the game; there are many other factors that come into play, but with having a set of stats in mind, Cal players and the team are able to control their controlables and fight to maximize their odds of winning. 

I have also included another function which places corners, at their average, which is around 5. Using this data, based on the number of saves Cal has, the number of shots on goal shifts, helping the coaching staff shift their stat goals as the game goes on. 

I hope that this model can be used to help our team grow stronger and build concrete goals to chase after each game!